# 00 — Build Dataset

Parses all raw data sources into three clean Parquet files that every downstream notebook reads from.

**Run this once (or when new data arrives). Outputs:**
- `data/sessions.parquet` — one row per workout
- `data/records.parquet` — per-second time series for all workouts
- `data/daily_signals.parquet` — one row per day (ATL/CTL/TSB, HRV, sleep, stress)

**Sources:**
- Part2 `.FIT` ZIP → workouts + per-second records
- Wellness JSONs → HRV, sleep, RHR
- Part1 `.FIT` ZIP via `fitfile` → intra-day stress + respiration

In [ ]:
import io, json, os, sys, zipfile, warnings, tempfile
from datetime import datetime

import fitparse
import fitfile
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────────────────
# Copy notebooks/config.example.py → notebooks/config.py and fill in your paths.
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import config

ACTIVITY_ZIP   = os.path.join(config.GARMIN_EXPORT_ROOT, 'DI-Connect-Uploaded-Files', 'UploadedFiles_0-_Part2.zip')
MONITORING_ZIP = os.path.join(config.GARMIN_EXPORT_ROOT, 'DI-Connect-Uploaded-Files', 'UploadedFiles_0-_Part1.zip')
WELLNESS_DIR   = os.path.join(config.GARMIN_EXPORT_ROOT, 'DI-Connect-Wellness')
DATA_DIR       = config.DATA_DIR

SEMI_TO_DEG = 180.0 / (2**31)
MAX_HR      = 190  # ← update to your actual max HR before running
ATL_TAU     = 7
CTL_TAU     = 42

os.makedirs(DATA_DIR, exist_ok=True)
print('Config loaded.')
print(f'  ACTIVITY_ZIP : {ACTIVITY_ZIP}')
print(f'  WELLNESS_DIR : {WELLNESS_DIR}')
print(f'  DATA_DIR     : {DATA_DIR}')

## 1. Parse Activity .FIT files → sessions + records

In [6]:
def strip_tz(dt):
    return dt.replace(tzinfo=None) if dt and hasattr(dt, 'tzinfo') and dt.tzinfo else dt

def session_get(session, *names):
    for n in names:
        try:
            v = session.get_value(n)
            if v is not None: return v
        except: pass
    return None

def parse_session(fit) -> dict | None:
    sessions = list(fit.get_messages('session'))
    if not sessions: return None
    s = sessions[0]
    sg = lambda *n: session_get(s, *n)

    start = strip_tz(sg('start_time'))
    if not start: return None
    elapsed = sg('total_elapsed_time') or 0

    return {
        'sport':        str(sg('sport') or 'unknown'),
        'start_time':   start,
        'date':         pd.Timestamp(start.date()),
        'duration_s':   float(elapsed),
        'distance_m':   float(sg('total_distance') or 0),
        'total_ascent': float(sg('total_ascent') or 0),
        'total_descent':float(sg('total_descent') or 0),
        'avg_hr':       sg('avg_heart_rate'),
        'max_hr':       sg('max_heart_rate'),
        'calories':     sg('total_calories'),
        'avg_cadence':  sg('avg_running_cadence') or sg('avg_cadence'),
        'avg_power':    sg('avg_power'),
        'norm_power':   sg('normalized_power'),
        'avg_vo':       sg('avg_vertical_oscillation'),
        'avg_stance':   sg('avg_stance_time'),
        'avg_step_len': sg('avg_step_length'),
        'avg_vr':       sg('avg_vertical_ratio'),
        'training_effect': sg('total_training_effect'),
        'anaerobic_effect': sg('total_anaerobic_training_effect'),
    }

def parse_records(fit, session_id: int) -> list[dict]:
    rows = []
    prev_alt = prev_dist = None
    for rec in fit.get_messages('record'):
        def rv(f):
            try: return rec.get_value(f)
            except: return None

        ts = strip_tz(rv('timestamp'))
        if not ts: continue

        lat = rv('position_lat')
        lon = rv('position_long')
        alt  = rv('enhanced_altitude') or rv('altitude')
        dist = rv('distance')
        spd  = rv('enhanced_speed') or rv('speed')
        pace = (1000 / float(spd) / 60) if spd else None

        raw_cad = rv('cadence')
        frac    = rv('fractional_cadence')
        cadence = ((float(raw_cad) + float(frac or 0)) * 2) if raw_cad is not None else None

        vo = rv('vertical_oscillation')
        if vo: vo = vo / 10.0  # mm → cm

        grad = None
        if alt is not None and prev_alt is not None:
            d_alt  = float(alt) - prev_alt
            d_dist = (float(dist) - prev_dist) if (dist and prev_dist) else (1000/(pace*60) if pace else None)
            if d_dist and d_dist > 0.5:
                grad = round(d_alt / d_dist * 100, 2)
        if alt  is not None: prev_alt  = float(alt)
        if dist is not None: prev_dist = float(dist)

        rows.append({
            'session_id':          session_id,
            'timestamp':           ts,
            'heart_rate':          rv('heart_rate'),
            'speed_ms':            float(spd) if spd else None,
            'pace_min_km':         pace,
            'cadence':             cadence,
            'altitude':            float(alt) if alt else None,
            'distance':            float(dist) if dist else None,
            'power':               rv('power'),
            'lat':                 float(lat) * SEMI_TO_DEG if lat else None,
            'lon':                 float(lon) * SEMI_TO_DEG if lon else None,
            'vertical_oscillation':vo,
            'vertical_ratio':      rv('vertical_ratio'),
            'stance_time':         rv('stance_time'),
            'step_length':         rv('step_length'),
            'gradient_pct':        grad,
        })
    return rows

print('Parsers defined.')

Parsers defined.


In [7]:
all_sessions = []
all_records  = []
session_id   = 0
skipped      = 0

with zipfile.ZipFile(ACTIVITY_ZIP) as zf:
    fit_files = [f for f in zf.namelist() if f.lower().endswith('.fit')]
    print(f'Parsing {len(fit_files)} .FIT files...')

    for fname in fit_files:
        data = zf.read(fname)
        try:
            fit1 = fitparse.FitFile(io.BytesIO(data))
            sess = parse_session(fit1)
            if not sess:
                skipped += 1
                continue

            fit2    = fitparse.FitFile(io.BytesIO(data))
            recs    = parse_records(fit2, session_id)
            if not recs:
                skipped += 1
                continue

            sess['session_id'] = session_id
            sess['n_records']  = len(recs)
            all_sessions.append(sess)
            all_records.extend(recs)
            session_id += 1
        except Exception as e:
            skipped += 1

df_sessions = pd.DataFrame(all_sessions).sort_values('start_time').reset_index(drop=True)
df_records  = pd.DataFrame(all_records)

print(f'Sessions parsed : {len(df_sessions)} (skipped {skipped})')
print(f'Record rows     : {len(df_records):,}')
print(f'Date range      : {df_sessions["date"].min().date()} → {df_sessions["date"].max().date()}')
print(f'Sports          : {df_sessions["sport"].value_counts().to_dict()}')

Parsing 787 .FIT files...
Sessions parsed : 38 (skipped 749)
Record rows     : 83,459
Date range      : 2026-02-26 → 2026-04-15
Sports          : {'training': 16, 'cycling': 7, 'rock_climbing': 6, 'running': 5, 'alpine_skiing': 2, 'soccer': 1, '64': 1}


## 2. Compute TRIMP and attach to sessions

In [8]:
ZONE_BOUNDS  = [0.50, 0.60, 0.70, 0.80, 0.90, 1.01]
ZONE_WEIGHTS = [1.0, 1.5, 2.0, 3.0, 4.0]

def compute_trimp(session_id: int) -> float:
    hr = df_records.loc[df_records['session_id'] == session_id, 'heart_rate'].dropna()
    if hr.empty: return 0.0
    trimp = 0.0
    for i, w in enumerate(ZONE_WEIGHTS):
        lo = ZONE_BOUNDS[i]  * MAX_HR
        hi = ZONE_BOUNDS[i+1] * MAX_HR
        trimp += ((hr >= lo) & (hr < hi)).sum() / 60 * w
    return trimp

print('Computing TRIMP for each session...')
df_sessions['trimp'] = df_sessions['session_id'].apply(compute_trimp)

# Fallback: sessions with no HR data get duration-based estimate
no_hr = df_sessions['trimp'] == 0
df_sessions.loc[no_hr, 'trimp'] = df_sessions.loc[no_hr, 'duration_s'] / 60 * 2.5

print(f'Sessions with HR data : {(~no_hr).sum()}')
print(f'TRIMP stats           : mean={df_sessions["trimp"].mean():.1f}, max={df_sessions["trimp"].max():.1f}')

Computing TRIMP for each session...
Sessions with HR data : 38
TRIMP stats           : mean=56.5, max=117.7


## 3. Build daily signals: ATL / CTL / TSB

In [9]:
daily_trimp = df_sessions.groupby('date')['trimp'].sum().reset_index()
full_range  = pd.DataFrame({'date': pd.date_range(df_sessions['date'].min(), pd.Timestamp.today())})
daily       = full_range.merge(daily_trimp, on='date', how='left').fillna(0)

a_atl = 1 - np.exp(-1 / ATL_TAU)
a_ctl = 1 - np.exp(-1 / CTL_TAU)

atl = ctl = 0.0
atl_vals, ctl_vals = [], []
for t in daily['trimp']:
    atl = a_atl * t + (1 - a_atl) * atl
    ctl = a_ctl * t + (1 - a_ctl) * ctl
    atl_vals.append(atl)
    ctl_vals.append(ctl)

daily['atl']  = atl_vals
daily['ctl']  = ctl_vals
daily['tsb']  = daily['ctl'] - daily['atl']
daily['acwr'] = (daily['atl'] / daily['ctl'].replace(0, np.nan)).fillna(0)

print(f'Daily frame: {len(daily)} days')
print(f'Current → ATL: {daily["atl"].iloc[-1]:.1f}, CTL: {daily["ctl"].iloc[-1]:.1f}, TSB: {daily["tsb"].iloc[-1]:.1f}')

Daily frame: 54 days
Current → ATL: 19.2, CTL: 25.7, TSB: 6.5


## 4. Load HRV + RHR + sleep from wellness JSONs

In [10]:
# HRV / RHR / respiration from healthStatusData
hrv_rows = []
for hf in sorted(f for f in os.listdir(WELLNESS_DIR) if 'healthStatus' in f):
    with open(os.path.join(WELLNESS_DIR, hf)) as f:
        for rec in json.load(f):
            row = {'date': pd.Timestamp(rec['calendarDate'])}
            for m in rec.get('metrics', []):
                t, v = m.get('type'), m.get('value')
                if   t == 'HRV':         row.update({'hrv': v, 'hrv_status': m.get('status'),
                                                     'hrv_baseline_low':  m.get('baselineLowerLimit'),
                                                     'hrv_baseline_high': m.get('baselineUpperLimit')})
                elif t == 'HR':          row['rhr'] = v
                elif t == 'RESPIRATION': row['daily_respiration'] = v
            hrv_rows.append(row)

df_hrv = pd.DataFrame(hrv_rows).sort_values('date').dropna(subset=['hrv']).reset_index(drop=True)

# Rolling HRV z-score (30-day window)
df_hrv['hrv_mean30'] = df_hrv['hrv'].rolling(30, min_periods=7).mean()
df_hrv['hrv_std30']  = df_hrv['hrv'].rolling(30, min_periods=7).std()
df_hrv['hrv_z']      = (df_hrv['hrv'] - df_hrv['hrv_mean30']) / df_hrv['hrv_std30'].replace(0, np.nan)

print(f'HRV records   : {len(df_hrv)}')
print(f'HRV range     : {df_hrv["hrv"].min():.0f}–{df_hrv["hrv"].max():.0f} ms  (mean {df_hrv["hrv"].mean():.0f})')

# Sleep from sleepData
sleep_rows = []
for sf in sorted(f for f in os.listdir(WELLNESS_DIR) if 'sleepData' in f):
    with open(os.path.join(WELLNESS_DIR, sf)) as f:
        for rec in json.load(f):
            if 'calendarDate' not in rec: continue
            sc = rec.get('sleepScores') or {}
            sleep_rows.append({
                'date':           pd.Timestamp(rec['calendarDate']),
                'sleep_duration_h': (rec.get('deepSleepSeconds',0)+rec.get('lightSleepSeconds',0)+rec.get('remSleepSeconds',0))/3600,
                'deep_h':         rec.get('deepSleepSeconds',0)/3600,
                'rem_h':          rec.get('remSleepSeconds',0)/3600,
                'sleep_score':    sc.get('overallScore'),
                'recovery_score': sc.get('recoveryScore'),
                'sleep_stress':   rec.get('avgSleepStress'),
                'sleep_respiration': rec.get('averageRespiration'),
            })

df_sleep = pd.DataFrame(sleep_rows).dropna(subset=['sleep_score']).sort_values('date').reset_index(drop=True)
print(f'Sleep records : {len(df_sleep)}  (avg score {df_sleep["sleep_score"].mean():.0f})')

HRV records   : 181
HRV range     : 19–98 ms  (mean 75)
Sleep records : 497  (avg score 74)


## 5. Load intra-day stress + respiration from Part1 .FIT files

In [11]:
stress_rows = []
respiration_rows = []

with zipfile.ZipFile(MONITORING_ZIP) as zf:
    mon_files = [f for f in zf.namelist() if f.lower().endswith('.fit')]
    print(f'Processing {len(mon_files)} monitoring files...')

    for fname in mon_files:
        data = zf.read(fname)
        try:
            with tempfile.NamedTemporaryFile(suffix='.fit', delete=False) as tmp:
                tmp.write(data); tmp_path = tmp.name
            try:
                ff = fitfile.file.File(tmp_path)
                if ff.type != fitfile.FileType.monitoring_b:
                    continue
                for msg in ff.messages:
                    fvs = msg.field_values
                    mtype = str(msg._DataMessage__definition_message.message_type)

                    if mtype == 'MessageType.stress_level':
                        ts  = fvs.get('local_timestamp')
                        lvl = fvs.get('stress_level')
                        if ts and lvl is not None:
                            ts_val = ts.value if hasattr(ts, 'value') else ts
                            lv_val = lvl.value if hasattr(lvl, 'value') else lvl
                            if isinstance(ts_val, datetime) and lv_val is not None:
                                stress_rows.append({'timestamp': strip_tz(ts_val), 'stress': float(lv_val)})

                    elif mtype == 'MessageType.respiration':
                        ts  = fvs.get('timestamp')
                        rr  = fvs.get('respiration_rate')
                        if ts and rr is not None:
                            ts_val = ts.value if hasattr(ts, 'value') else ts
                            rr_val = rr.value if hasattr(rr, 'value') else rr
                            if isinstance(ts_val, datetime) and rr_val is not None:
                                respiration_rows.append({'timestamp': strip_tz(ts_val), 'respiration': float(rr_val)})
            finally:
                os.unlink(tmp_path)
        except: pass

df_stress = pd.DataFrame(stress_rows).sort_values('timestamp').reset_index(drop=True) if stress_rows else pd.DataFrame()
df_resp   = pd.DataFrame(respiration_rows).sort_values('timestamp').reset_index(drop=True) if respiration_rows else pd.DataFrame()

print(f'Stress readings     : {len(df_stress):,}')
print(f'Respiration readings: {len(df_resp):,}')

# Aggregate to daily averages
if not df_stress.empty:
    df_stress['date'] = pd.to_datetime(df_stress['timestamp'].dt.date)
    daily_stress = df_stress.groupby('date')['stress'].mean().reset_index()
    daily_stress.columns = ['date', 'avg_stress']
else:
    daily_stress = pd.DataFrame(columns=['date', 'avg_stress'])

if not df_resp.empty:
    df_resp['date'] = pd.to_datetime(df_resp['timestamp'].dt.date)
    daily_resp = df_resp.groupby('date')['respiration'].mean().reset_index()
    daily_resp.columns = ['date', 'avg_respiration']
else:
    daily_resp = pd.DataFrame(columns=['date', 'avg_respiration'])

print(f'Stress days covered    : {len(daily_stress)}')
print(f'Respiration days covered: {len(daily_resp)}')

Processing 7288 monitoring files...
Stress readings     : 0
Respiration readings: 0
Stress days covered    : 0
Respiration days covered: 0


## 6. Merge into daily_signals

In [12]:
daily_signals = daily.copy()
daily_signals = daily_signals.merge(df_hrv[['date','hrv','hrv_z','hrv_status','rhr','hrv_baseline_low','hrv_baseline_high']], on='date', how='left')
daily_signals = daily_signals.merge(df_sleep[['date','sleep_duration_h','sleep_score','recovery_score','sleep_stress','deep_h','rem_h']], on='date', how='left')
daily_signals = daily_signals.merge(daily_stress, on='date', how='left')
daily_signals = daily_signals.merge(daily_resp,   on='date', how='left')

print(f'daily_signals shape : {daily_signals.shape}')
print(f'\nNull rates (%):')
nulls = (daily_signals.isnull().mean() * 100).round(1)
print(nulls[nulls > 0].sort_values(ascending=False).to_string())

daily_signals shape : (54, 20)

Null rates (%):
avg_respiration      100.0
avg_stress           100.0
hrv_status            13.0
rhr                   13.0
hrv                   13.0
hrv_z                 13.0
hrv_baseline_high     13.0
hrv_baseline_low      13.0
sleep_duration_h      13.0
sleep_score           13.0
sleep_stress          13.0
recovery_score        13.0
rem_h                 13.0
deep_h                13.0


## 7. Save to Parquet

In [13]:
sessions_path = os.path.join(DATA_DIR, 'sessions.parquet')
records_path  = os.path.join(DATA_DIR, 'records.parquet')
daily_path    = os.path.join(DATA_DIR, 'daily_signals.parquet')

df_sessions.to_parquet(sessions_path, index=False)
df_records.to_parquet(records_path,   index=False)
daily_signals.to_parquet(daily_path,  index=False)

print('Saved:')
for path, df in [(sessions_path, df_sessions), (records_path, df_records), (daily_path, daily_signals)]:
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {os.path.basename(path):30s} {len(df):>8,} rows  {size_mb:.1f} MB')

Saved:
  sessions.parquet                     38 rows  0.0 MB
  records.parquet                  83,459 rows  1.4 MB
  daily_signals.parquet                54 rows  0.0 MB


## 8. Verification

In [14]:
print('=== sessions.parquet ===')
s = pd.read_parquet(sessions_path)
print(f'  Shape      : {s.shape}')
print(f'  Date range : {s["date"].min().date()} → {s["date"].max().date()}')
print(f'  Sports     : {s["sport"].value_counts().to_dict()}')
print(f'  TRIMP      : mean={s["trimp"].mean():.1f}, max={s["trimp"].max():.1f}')
print()

print('=== records.parquet ===')
r = pd.read_parquet(records_path)
print(f'  Shape      : {r.shape}')
print(f'  Columns    : {list(r.columns)}')
running_recs = r[r['session_id'].isin(s[s['sport'].isin(['running','trail_running'])]['session_id'])]
print(f'  Running records (HR, cadence, VO, stance): {len(running_recs):,}')
print()

print('=== daily_signals.parquet ===')
d = pd.read_parquet(daily_path)
print(f'  Shape      : {d.shape}')
print(f'  Date range : {d["date"].min().date()} → {d["date"].max().date()}')
last = d.dropna(subset=['atl']).iloc[-1]
print(f'  Last day   : ATL={last["atl"]:.1f}, CTL={last["ctl"]:.1f}, TSB={last["tsb"]:.1f}')
print(f'  HRV days   : {d["hrv"].notna().sum()}')
print(f'  Sleep days : {d["sleep_score"].notna().sum()}')
print(f'  Stress days: {d["avg_stress"].notna().sum()}')
print()
print('✓ Dataset ready. Open notebooks/01_eda_training_signals.ipynb to explore.')

=== sessions.parquet ===
  Shape      : (38, 22)
  Date range : 2026-02-26 → 2026-04-15
  Sports     : {'training': 16, 'cycling': 7, 'rock_climbing': 6, 'running': 5, 'alpine_skiing': 2, 'soccer': 1, '64': 1}
  TRIMP      : mean=56.5, max=117.7

=== records.parquet ===
  Shape      : (83459, 16)
  Columns    : ['session_id', 'timestamp', 'heart_rate', 'speed_ms', 'pace_min_km', 'cadence', 'altitude', 'distance', 'power', 'lat', 'lon', 'vertical_oscillation', 'vertical_ratio', 'stance_time', 'step_length', 'gradient_pct']
  Running records (HR, cadence, VO, stance): 6,153

=== daily_signals.parquet ===
  Shape      : (54, 20)
  Date range : 2026-02-26 → 2026-04-20
  Last day   : ATL=19.2, CTL=25.7, TSB=6.5
  HRV days   : 47
  Sleep days : 47
  Stress days: 0

✓ Dataset ready. Open notebooks/01_eda_training_signals.ipynb to explore.
